In [1]:
import os
from cognite.client import CogniteClient, ClientConfig
from cognite.client.credentials import OAuthClientCredentials

In [ ]:
BASE_URL = "https://aw-was-gp-001.cognitedata.com"
PROJECT = "alimentospolarcomercialca"
TOKEN_URL = "https://datamosaix-prod.us.auth0.com/oauth/token"

# 1. OAuth Credentials configured for Auth0
creds = OAuthClientCredentials(
    token_url=TOKEN_URL,
    client_id=os.getenv("CLIENT_ID", "ccc"),
    client_secret=os.getenv("CLIENT_SECRET", "xxx"),
    audience="https://cognitedata.com",  # Required global audience for DataMosaix
    scopes=[]
)

In [3]:
# 2. Client Config
cnf = ClientConfig(
    client_name="superenvases-data-mosaix",
    project=PROJECT,
    credentials=creds,
    base_url=BASE_URL
)

In [4]:
# 3. Instantiate and Test Connection
try:
    client = CogniteClient(cnf)
    spaces = client.data_modeling.spaces.list()
    print("Successfully authenticated via Auth0!")
    print(f"Active spaces found in '{PROJECT}': {[s.space for s in spaces]}")
except Exception as e:
    print("Authentication error:")
    print(e)

Successfully authenticated via Auth0!
Active spaces found in 'alimentospolarcomercialca': []


In [5]:
client = CogniteClient(cnf)

In [17]:
from cognite.client.data_classes.data_modeling import (
    SpaceApply,
    NodeApply,
    EdgeApply,
    NodeOrEdgeData,  # <--- Added missing class
    DirectRelationReference,
    ViewId
)
# 2. Create the Space
CUSTOM_SPACE = "superenvases_plant"
space = SpaceApply(
    space=CUSTOM_SPACE,
    name="Superenvases Plant Space",
    description="Space for Planta Superenvases asset hierarchy and telemetry"
)

print(f"Creating space '{CUSTOM_SPACE}'...")
client.data_modeling.spaces.apply(space)
print("Space ready.")

Creating space 'superenvases_plant'...
Space ready.


In [24]:
# 2. Setup Definitions
cdm_asset_view = ViewId(space="cdf_cdm", external_id="CogniteAsset", version="v1")
edge_type = DirectRelationReference(space="cdf_cdm", external_id="has-parent")

nodes = []
edges = []

def add_asset(asset_id, name, desc, parent_id=None):
    properties = {
        "name": name,
        "description": desc
    }
    # Link hierarchy via the 'parent' Direct Relation property
    if parent_id:
        properties["parent"] = {
            "space": CUSTOM_SPACE,
            "externalId": parent_id
        }

    nodes.append(
        NodeApply(
            space=CUSTOM_SPACE,
            external_id=asset_id,
            sources=[
                NodeOrEdgeData(
                    source=cdm_asset_view,
                    properties=properties
                )
            ]
        )
    )

In [25]:
# --- Root Plant ---
add_asset("facility_planta_superenvases", "Planta Superenvases", "Main manufacturing plant")

In [26]:
# --- Production Lines ---
add_asset("line_l1", "Line L1", "Production Line 1", "facility_planta_superenvases")
add_asset("line_l3", "Line L3", "Production Line 3", "facility_planta_superenvases")

In [27]:
# --- Line 1 Machines ---
add_asset("machine_minster_l1", "MINSTER L1", "Minster press on Line 1", "line_l1")
add_asset("machine_printer_l1", "PRINTER L1", "Printer system on Line 1", "line_l1")

for i in [11, 12, 14, 15, 17, 18]:
    add_asset(f"machine_di{i}", f"D&I{i}", f"D&I machine {i} on Line 1", "line_l1")

for i in range(11, 16):
    add_asset(f"machine_ispray{i}", f"ISPRAY {i}", f"Internal spray {i} on Line 1", "line_l1")

In [28]:
# --- Line 3 Machines ---
add_asset("machine_minster_l3", "MINSTER L3", "Minster press on Line 3", "line_l3")
add_asset("machine_printer31", "PRINTER 31", "Printer 31 on Line 3", "line_l3")
add_asset("machine_printer32", "PRINTER 32", "Printer 32 on Line 3", "line_l3")

for i in range(31, 39):
    add_asset(f"machine_standum{i}", f"STANDUM {i}", f"Standum machine {i} on Line 3", "line_l3")

for i in range(31, 39):
    add_asset(f"machine_ispray{i}", f"ISPRAY {i}", f"Internal spray {i} on Line 3", "line_l3")

In [29]:
# 3. Deploy Hierarchy to Data Mosaix
print(f"Deploying {len(nodes)} assets and {len(edges)} hierarchy connections...")
response = client.data_modeling.instances.apply(nodes=nodes, edges=edges)
print(f"Deployment complete! Successfully applied {len(response.nodes)} nodes and {len(response.edges)} edges.")

Deploying 35 assets and 0 hierarchy connections...
Deployment complete! Successfully applied 35 nodes and 34 edges.


In [30]:
from cognite.client.data_classes.data_modeling import ViewId

# 1. Fetch nodes from your space
asset_view = ViewId(space="cdf_cdm", external_id="CogniteAsset", version="v1")

nodes = client.data_modeling.instances.list(
    instance_type="node",
    space="superenvases_plant",
    sources=[asset_view],
    limit=100
)

print(f"Retrieved {len(nodes)} nodes from space 'superenvases_plant':\n")
print(f"{'EXTERNAL ID':<32} | {'NAME':<20} | {'PARENT EXTERNAL ID'}")
print("-" * 75)

for node in nodes:
    # Read properties stored under the CogniteAsset view
    props = node.properties.get(asset_view, {})
    name = props.get("name", "N/A")
    parent_ref = props.get("parent")
    parent_id = parent_ref.get("externalId") if isinstance(parent_ref, dict) else "None (Root)"
    
    print(f"{node.external_id:<32} | {name:<20} | {parent_id}")

Retrieved 35 nodes from space 'superenvases_plant':

EXTERNAL ID                      | NAME                 | PARENT EXTERNAL ID
---------------------------------------------------------------------------
facility_planta_superenvases     | Planta Superenvases  | None (Root)
line_l1                          | Line L1              | facility_planta_superenvases
line_l3                          | Line L3              | facility_planta_superenvases
machine_di11                     | D&I11                | line_l1
machine_di12                     | D&I12                | line_l1
machine_di14                     | D&I14                | line_l1
machine_di15                     | D&I15                | line_l1
machine_di17                     | D&I17                | line_l1
machine_di18                     | D&I18                | line_l1
machine_ispray11                 | ISPRAY 11            | line_l1
machine_ispray12                 | ISPRAY 12            | line_l1
machine_ispray13      